In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.stats import chi2
from dataclasses import asdict
from joblib import Parallel, delayed
from datetime import datetime

from core import (
    Config, Simulator, NumpyEncoder,
    save_results, load_results,
    get_model_description, create_shared_experiment,
    build_sim_kwargs, compute_beta_from_ratio,
    setup_plot_style, STRATEGY_STYLES, XLABEL_LOG_RATIO,
)


def _run_single_vr(cfg, shared, total_signal_std, target_delta, seed):

    np.random.seed(seed)
    std_pre, std_trt = shared['std_pre'], shared['std_trt']
    beta_pre_val, beta_trt_val = compute_beta_from_ratio(target_delta, std_pre, std_trt, total_signal_std)

    actual_ratio = (beta_trt_val * std_trt) / (beta_pre_val * std_pre) if beta_pre_val * std_pre > 0 else float('inf')
    actual_log_ratio = np.log2(actual_ratio) if actual_ratio > 0 and np.isfinite(actual_ratio) else target_delta

    sim = Simulator(**build_sim_kwargs(cfg, shared, beta_pre_val, beta_trt_val))


    ps, pd = sim.p_pre, sim.p_dynamic
    taus = []
    for _ in range(2000):
        Z = np.random.binomial(1, cfg.p_treat, sim.n)
        if 0 < Z.sum() < sim.n:
            X = np.hstack([sim.X_pre, sim._get_covs(Z, cfg.rerand_covs)])
            taus.append(sim._hajek(Z, X))
    taus = np.array(taus)

    def safe_inv(C, d):
        if d > 1: return np.linalg.inv(C + 1e-6 * np.eye(d))
        return np.array([[1 / (float(C) + 1e-6)]])

    sigma_inv = {}
    if ps > 0:
        sigma_inv['static'] = safe_inv(np.cov(taus[:, :ps], rowvar=False), ps)
    if pd > 0:
        sigma_inv['dynamic'] = safe_inv(np.cov(taus[:, ps:], rowvar=False), pd)
    if ps + pd > 0:
        sigma_inv['all'] = safe_inv(np.cov(taus, rowvar=False), ps + pd)

    thresh = {
        'static': chi2.ppf(cfg.acceptance_rate, ps) if ps > 0 else 0,
        'dynamic': chi2.ppf(cfg.acceptance_rate, pd) if pd > 0 else 0,
        'all': chi2.ppf(cfg.acceptance_rate, ps + pd) if ps + pd > 0 else 0,
    }

    strategies = ['static', 'dynamic', 'all']
    results = {m: [] for m in ['raw'] + strategies}
    needed = cfg.n_accepted

    while not all(len(results[m]) >= needed for m in strategies) or len(results['raw']) < needed * 5:
        Z = np.random.binomial(1, cfg.p_treat, sim.n)
        if not (0 < Z.sum() < sim.n): continue

        X_dyn = sim._get_covs(Z, cfg.rerand_covs)
        tau = sim._hajek(Z, np.hstack([sim.X_pre, X_dyn]))

        accept = {}
        if 'static' in strategies and len(results['static']) < needed:
            M = tau[:ps] @ sigma_inv['static'] @ tau[:ps]
            accept['static'] = M <= thresh['static']
        if 'dynamic' in strategies and len(results['dynamic']) < needed:
            M = tau[ps:] @ sigma_inv['dynamic'] @ tau[ps:]
            accept['dynamic'] = M <= thresh['dynamic']
        if 'all' in strategies and len(results['all']) < needed:
            M = tau @ sigma_inv['all'] @ tau
            accept['all'] = M <= thresh['all']

        if not any(accept.values()) and len(results['raw']) >= needed * 5: continue

        tau_y = sim._hajek(Z, sim._outcome(Z).reshape(-1, 1))[0]
        if len(results['raw']) < needed * 5:
            results['raw'].append(tau_y)
        for m, ok in accept.items():
            if ok:
                results[m].append(tau_y)

    var_raw = np.var(results['raw'])
    vr = {m: np.var(results[m]) / var_raw if var_raw > 0 else 1.0 for m in strategies}

    return {
        'target_log_ratio': float(target_delta),
        'actual_log_ratio': float(actual_log_ratio),
        'beta_pre': float(beta_pre_val), 'beta_trt': float(beta_trt_val),
        'var_raw': float(var_raw),
        'variance_ratios': {m: float(vr[m]) for m in strategies},
    }


def run_vr_experiment(cfg, target_log_ratios=None, total_signal_std=30.0, n_jobs=-1):

    if target_log_ratios is None:
        target_log_ratios = np.linspace(-4, 4, 9)

    shared = create_shared_experiment(cfg)
    print(f"Outcome Type {cfg.outcome_type} | Rerand covs: {cfg.rerand_covs}")
    print(f"Block stds: static={shared['std_pre']:.4f}, dynamic={shared['std_trt']:.4f}")
    print(f"Running {len(target_log_ratios)} sweep points...\n")

    sweep = Parallel(n_jobs=n_jobs, verbose=5)(
        delayed(_run_single_vr)(cfg, shared, total_signal_std, float(d), cfg.seed + i)
        for i, d in enumerate(target_log_ratios)
    )
    sweep.sort(key=lambda x: x['target_log_ratio'])

    results = {
        'config': asdict(cfg),
        'model': {'outcome_type': cfg.outcome_type, 'description': shared['description']},
        'block_stds': {'std_pre': shared['std_pre'], 'std_trt': shared['std_trt']},
        'total_signal_std': total_signal_std,
        'sweep': sweep,
        'timestamp': datetime.now().isoformat(),
    }

    for s in sweep:
        vr = s['variance_ratios']
        print(f"  log2(R)={s['actual_log_ratio']:6.2f} | "
              f"VRatio: pre-treat={vr['static']:.3f}, "
              f"treat-dep={vr['dynamic']:.3f}, "
              f"joint={vr['all']:.3f}")
    return results


def plot_vr(results, save_path=None):


    setup_plot_style()
    ratios = [s['actual_log_ratio'] for s in results['sweep']]
    vratios = {
        m: [s['variance_ratios'][m] for s in results['sweep']]
        for m in ['static', 'dynamic', 'all']
    }

    fig, ax = plt.subplots(figsize=(12, 9))
    for m, (color, marker, label) in STRATEGY_STYLES.items():
        ax.plot(ratios, vratios[m], f'{marker}-', color=color, lw=3.0,
                label=label, alpha=0.9, ms=10)

    ax.set_xlabel(XLABEL_LOG_RATIO, labelpad=15)
    ax.set_ylabel('Variance Ratio', fontweight='bold', labelpad=15)
    ax.set_xlim(-4, 4)
    ax.axvline(0, color='gray', ls=':', alpha=0.8, lw=1.5)
    ax.axhline(1.0, color='gray', ls='--', alpha=0.6, lw=1.5)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend(frameon=True, framealpha=0.95, fancybox=True, loc='best')

    plt.subplots_adjust(top=0.90, bottom=0.12, left=0.12, right=0.95)

    if save_path: plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


print('Loaded: run_variance_reduction')


In [ ]:
import os, traceback

EXPERIMENTS = [
    (1, ['neighbor_prop']),
    (1, ['neighbor_weighted_X']),
    (1, ['neighbor_prop', 'neighbor_weighted_X']),
    (2, ['neighbor_prop']),
    (2, ['neighbor_weighted_X']),
    (2, ['neighbor_prop', 'neighbor_weighted_X']),
    (3, ['neighbor_prop']),
    (3, ['neighbor_weighted_X']),
    (3, ['neighbor_prop', 'neighbor_weighted_X']),
    (5, ['neighbor_prop']),
    (5, ['neighbor_weighted_X']),
    (5, ['neighbor_prop', 'neighbor_weighted_X']),
    (6, ['neighbor_prop']),
    (6, ['neighbor_weighted_X']),
    (6, ['neighbor_prop', 'neighbor_weighted_X']),
    (7, ['neighbor_prop']),
    (7, ['neighbor_weighted_X']),
    (7, ['neighbor_prop', 'neighbor_weighted_X']),
    (8, ['neighbor_prop']),
    (8, ['neighbor_weighted_X']),
    (8, ['neighbor_prop', 'neighbor_weighted_X']),
]

_COVS_SHORT = {
    'neighbor_prop': 'np',
    'neighbor_weighted_X': 'nwx',
    'second_order_neighbor_weighted_X': 'sonwx',
}
def covs_short(covs):
    return '+'.join(_COVS_SHORT.get(c, c) for c in covs)

os.makedirs('results_vr_data', exist_ok=True)
os.makedirs('results_vr_picture', exist_ok=True)
LOG_PATH = 'results_vr_data/vr_batch_log.txt'
with open(LOG_PATH, 'w') as logf:
    logf.write(f'Batch VR started: {datetime.now().isoformat()}\n')
    logf.write(f'Total experiments: {len(EXPERIMENTS)}\n\n')

for i, (otype, covs) in enumerate(EXPERIMENTS, 1):
    sn = covs_short(covs)
    tag = f'type{otype}_covs-{sn}'
    print(f'\n[{i}/{len(EXPERIMENTS)}] {tag} | covs={covs}')
    t0 = datetime.now()
    try:
        cfg = Config(
            n_units=2000,
            outcome_type=otype,
            n_pre=6,
            n_hidden=1,
            rerand_covs=covs,
            acceptance_rate=0.05,
            n_accepted=2000,
            p_treat=0.5,


            graph_type='ba',
            graph_m=5,


        )
        results = run_vr_experiment(
            cfg,
            target_log_ratios=np.linspace(-4, 4, 9),
            total_signal_std=30.0,
            n_jobs=1,
        )
        save_results(results, save_dir='results_vr_data', prefix=f'vr_covs-{sn}_')


        pdf_path = f'results_vr_picture/vr_covs-{sn}_type{otype}.pdf'
        plot_vr(results, save_path=pdf_path)
        plt.close('all')
        dt = (datetime.now() - t0).total_seconds()
        status = f'OK ({dt:.0f}s) -> {pdf_path}'
    except Exception as e:
        status = f'FAILED: {e!r}'
        traceback.print_exc()
    print(f'  [{tag}] {status}')
    with open(LOG_PATH, 'a') as logf:
        logf.write(f'[{datetime.now().isoformat()}] {tag} | {status}\n')

print('\n=== Batch VR complete ===')
with open(LOG_PATH, 'a') as logf:
    logf.write(f'\nBatch VR complete: {datetime.now().isoformat()}\n')
